# Train the watermark (objectives 1 and 3)

This notebook trains the StegaStamp-style encoder and decoder.

* **Objective 1.** The stamped 400×400 file decodes on its own.
* **Objective 3.** The same file still decodes after brightness, contrast, JPEG, and a small border crop (up to about 10% off each edge). The signal is pushed toward the center by the edge penalty, and training includes those crops.

The finder that locates a stamp inside a 1000×900 canvas is **not** trained here. Build that dataset later, from stamps this run produces.

Upload the whole `own` folder (it contains `stegwatermark/`) to Google Drive before running the real training cell. A T4 runtime is enough. Checkpoints go to Drive and include the optimizer and the step counter, so a disconnect can be resumed.

MIRFLICKR photos are mixed sizes. The loader fits each one to 400×400. They are not already 400×400.


## 1. Runtime and libraries

Use a GPU runtime. This does not reinstall PyTorch if Colab already has it.

In [ ]:
import importlib.util
if importlib.util.find_spec("torch") is None:
    %pip install -q torch torchvision
%pip install -q lpips opencv-python-headless pillow bchlib


## 2. Drive and project folder

Mount Drive first. `PROJECT_ROOT` must be the directory that contains `stegwatermark/`.

In [ ]:
from pathlib import Path
import sys

DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OK = True
except Exception as error:
    print("Drive not mounted (%s)." % error)

CANDIDATES = [
    Path("/content/drive/MyDrive/own"),
    Path("/content/own"),
    Path.cwd(),
    Path.cwd().parent,
]
PROJECT_ROOT = None
for path in CANDIDATES:
    if (path / "stegwatermark").is_dir():
        PROJECT_ROOT = path
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find stegowatermark/. Upload the own folder to Drive and set PROJECT_ROOT."
    )
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT", PROJECT_ROOT)


## 3. Data and checkpoint directory

Mount Drive, then set `DATA_DIR` to the unzipped MIRFLICKR image folder. Copying it to `/content/data` makes the loader faster than reading every JPEG from Drive.

The real run is 140k steps, batch size 4. That is several hours on a T4. The smoke cell below only checks that one full forward/backward works.


In [ ]:
DATA_DIR = "/content/data/mirflickr"          # folder of JPEG/PNG covers
SAVE_DIR = (
    "/content/drive/MyDrive/watermark_checkpoints"
    if DRIVE_OK
    else str(PROJECT_ROOT / "checkpoints")
)
print("DATA_DIR", DATA_DIR)
print("SAVE_DIR", SAVE_DIR)


## 4. Smoke test

Two steps on random images, with the image losses turned on, so the encoder, decoder, critic, warp, JPEG, and checkpoint all run once.

In [ ]:
from stegowatermark.train_loop import TrainConfig, run_training

smoke = TrainConfig(
    synthetic=True,
    num_steps=2,
    batch_size=1,
    no_im_loss_steps=0,
    save_every=1,
    log_every=1,
    save_dir="/tmp/watermark_smoke",
    device="cuda",
)
run_training(smoke)
print("smoke test finished")


## 5. Real training

`bit_acc` is measured after distortion and crop. `clean` is the same batch with no distortion. Clean accuracy should climb toward 0.95 first. Augmented accuracy follows as the ramps increase.

Loss weights match `scripts/base.sh`:

* steps 0–1500: message BCE only
* L2 (weight 2), LPIPS (weight 1.5), critic (weight 0.5) ramp over 15k steps
* Y/U/V scales 1 / 100 / 100
* warp ramps to ±10% of the width over 10k steps, empty borders filled white
* edge penalty starts at step 80k
* border-crop augmentation ramps to 10% after step 1500

To resume after a disconnect, set `RESUME` to `latest.pt`. The schedules continue from the saved step.


In [ ]:
RESUME = ""  # e.g. SAVE_DIR + "/latest.pt"

config = TrainConfig(
    data_dir=DATA_DIR,
    save_dir=SAVE_DIR,
    resume=RESUME,
    synthetic=False,
    batch_size=4,
    num_steps=140000,
    device="cuda",
)
run_training(config)


## What this checkpoint can and cannot do

It can stamp a 400×400 image and read that image back, including after the light edits in objective 3.

It cannot find the stamp inside a 1000×900 screenshot. That is the finder, trained afterwards by pasting these stamps onto large canvases.
